# Hardware-in-the-loop, explained

**Why you can trust that the chip runs what you simulated.**

Every other tutorial makes a claim like *"the compiled controller matches the
simulator bit-for-bit."* This one explains the machinery behind that claim — and,
crucially, shows it has **teeth**: the check *catches* code-generation bugs rather
than rubber-stamping them.

The idea of hardware-in-the-loop (HIL) parity is simple:

```
  host reference  ──┐
  (JAX simulation)  ├──►  step-locked compare  ──►  pass / FAIL  (+ where & why)
  target artifact ──┘        under a documented
  (the deployed C)            tolerance contract
```

Same inputs, same seed, same horizon. Run both, line them up cycle-by-cycle, and
compare each quantity against a **documented tolerance** for that
`(target family, precision, quantity)`. If they agree, the codegen is faithful. If
they don't, the report points at the offending quantity and step.

In [1]:
import os, platform, ctypes
from pathlib import Path
def _preload_acados():
    root = os.environ.get("ACADOS_SOURCE_DIR")
    if not root or platform.system() != "Darwin": return
    for n in ("libblasfeo.dylib","libhpipm.dylib","libqpOASES_e.dylib","libacados.dylib"):
        p = Path(root)/"lib"/n
        if p.exists():
            try: ctypes.CDLL(str(p))
            except OSError: pass
_preload_acados()
import numpy as np, jax.numpy as jnp
print("acados", "set" if os.environ.get("ACADOS_SOURCE_DIR") else "MISSING (build cells will fail)")

acados set


## Step 1 — The tolerance contract

There are no magic numbers. Every acceptable divergence is a **documented row** in
a table keyed by `(target_family, dtype, quantity)`, each with an *intent* grade:

- **`bit-exact`** — bit-identical (content hashes, integer-coded quantities).
- **`ulp-bounded`** — within a small ULP count; the embedded-default for correct
  codegen at a single precision.
- **`approximate`** — engineering tolerance, e.g. comparing a `float32` artifact
  against a `float64` reference.

The table *is* the contract — loosening a bound is a reviewed, documented change.

In [2]:
from jaxility.testing.tolerances import lookup_tolerance, quantities_for

fam = "cortex-a76"
print(f"quantities covered for {fam} x float64:", quantities_for(fam, "float64"))
print()
for q in quantities_for(fam, "float64"):
    t = lookup_tolerance(fam, "float64", q)
    print(f"  {q:18s} abs={t.abs_tol:.0e}  rel={t.rel_tol:.0e}  [{t.grade}]")
print("\n-> the lookup key is (target_family, dtype, quantity); a miss is a loud")
print("   error, never a silent default.")

quantities covered for cortex-a76 x float64: ['actuator_torque', 'joint_position', 'joint_velocity']

  actuator_torque    abs=1e-09  rel=1e-07  [ulp-bounded]
  joint_position     abs=1e-09  rel=1e-07  [ulp-bounded]
  joint_velocity     abs=1e-09  rel=1e-07  [ulp-bounded]

-> the lookup key is (target_family, dtype, quantity); a miss is a loud
   error, never a silent default.


## Step 2 — A clean parity run

Let's produce a real one. We build the Cartpole LQR controller, close the loop two
ways — a **host reference** (acados control + a JAX RK4 plant) and the **generated
C binary** (acados control + acados-sim plant) — and compare them step-locked.

In [3]:
import tempfile
import jaxterity.zoo as zoo
from jaxterity.zoo.cartpole import reduced_params
from jaxility.lowering import translate
from jaxility.templates import lqr
from jaxility.builder import build_for_target
from jaxility.targets import current_host_target
from jaxility.hil import (CARTPOLE_LQR_SCHEMA, CARTPOLE_LQR_TRACE, LocalRunner,
    build_controller_hil_binary, generate_controller_hil_source, run_hil)

NX, NU = 4, 1
INITIAL_STATE = (0.3, 0.0, 0.0, 0.0)
PLANT_DT = 1.0/20
robot = zoo.load("cartpole").with_provenance(("hil-tut","v0","tel"), calibrated=True)
p = reduced_params(robot)

def dyn(state, control):
    g, mp, mc, L = p["g"], p["mp"], p["mc"], p["L"]
    th, xd, thd = state[1], state[2], state[3]
    s, c = jnp.sin(th), jnp.cos(th); den = mc + mp*s*s
    return jnp.array([xd, thd, (control[0]+mp*s*(L*thd**2+g*c))/den,
        (-control[0]*c-mp*L*thd**2*c*s-(mc+mp)*g*s)/(L*den)])

cf = translate(dyn, in_shapes=((NX,),(NU,)), name="hil_cartpole")
spec = lqr(cf, Q=(10.,10.,1.,1.), R=(0.1,), initial_state=INITIAL_STATE,
           input_bounds=((-20.,),(20.,)), name="hil_cartpole", horizon_steps=20, time_horizon_s=1.0)
bundle = build_for_target(dynamics=cf, spec=spec, target=current_host_target(),
    source_attestation_handle=bytes.fromhex(robot.attestation_handle),
    work_dir=Path(tempfile.mkdtemp()))

def _rk4(x,u,dt):
    xj,uj=jnp.asarray(x),jnp.asarray(u)
    k1=dyn(xj,uj);k2=dyn(xj+0.5*dt*k1,uj);k3=dyn(xj+0.5*dt*k2,uj);k4=dyn(xj+dt*k3,uj)
    return np.asarray(xj+(dt/6.)*(k1+2*k2+2*k3+k4))

def host_reference(n, seed=0):
    bundle.solver.reset(); x=np.array(INITIAL_STATE); x[0]+=0.01*seed
    pos,vel,tau=np.empty((n,2)),np.empty((n,2)),np.empty((n,1))
    for i in range(n):
        bundle.solver.set(0,"lbx",x); bundle.solver.set(0,"ubx",x); bundle.solver.solve()
        u=np.asarray(bundle.solver.get(0,"u")); pos[i],vel[i],tau[i]=x[:2],x[2:],u
        x=_rk4(x,u,PLANT_DT)
    return {"joint_position":pos,"joint_velocity":vel,"actuator_torque":tau}

gen=bundle.shared_library_path.parent
exe=build_controller_hil_binary(generated_code_dir=gen, model_name="hil_cartpole",
    source=generate_controller_hil_source(model_name="hil_cartpole", nx=NX, nu=NU,
        trace=CARTPOLE_LQR_TRACE, initial_state=INITIAL_STATE), out_path=gen/"hil_cartpole_hil")
reference = host_reference(40)
report = run_hil(reference, LocalRunner(exe), target_family="cortex-a76", dtype="float64",
    schema=CARTPOLE_LQR_SCHEMA, n_steps=40, seed=0)
print("overall passed:", report.passed)
for q in report.equivalence.per_quantity:
    print(f"  {q.quantity:18s} max_abs={q.max_abs_error:.2e}  (bound {q.tolerance.abs_tol:.0e}, {q.tolerance.grade})")

overall passed: True
  actuator_torque    max_abs=4.22e-15  (bound 1e-09, ulp-bounded)
  joint_position     max_abs=5.55e-17  (bound 1e-09, ulp-bounded)
  joint_velocity     max_abs=4.44e-16  (bound 1e-09, ulp-bounded)


The divergence is ~1e-15 — the **plant-model mismatch floor** (acados' ERK
sim vs the JAX RK4 reference), orders of magnitude under the bound. The Python and
C code paths agree. That's what "faithful codegen" looks like.

## Step 3 — The violation rule (why it's robust)

A quantity fails only when **both** the absolute *and* the relative per-step error
exceed their bounds at the same step:

```
fail  ⟺  per_step_abs > abs_tol  AND  per_step_rel > rel_tol
```

This rejects two false alarms: a near-zero quantity whose *relative* error spikes
on noise (abs is tiny → not a failure), and a huge-magnitude quantity whose
*absolute* error is large but relatively trivial. The conjunction is what makes the
check trustworthy rather than flaky.

## Step 4 — Does it have teeth? Catch a codegen bug

A parity check is only worth anything if it *fails* when the generated code is
wrong. Let's simulate two classic code-generation bugs by corrupting the candidate
trajectory, and watch the harness catch each — naming the quantity, the first bad
step, and a structured hint.

In [4]:
from jaxility.testing.equivalence import compare

# Bug A: a wrong actuator gain — a constant offset on the actuator command.
buggy = {k: v.copy() for k, v in reference.items()}
buggy["actuator_torque"] = buggy["actuator_torque"] + 0.5   # 0.5 N that shouldn't be there
rep_a = compare(reference, buggy, target_family="cortex-a76", dtype="float64")
print("Bug A (wrong actuator gain): passed =", rep_a.overall_passed)
for q in rep_a.per_quantity:
    if not q.passed:
        print(f"  CAUGHT {q.quantity}: max_abs={q.max_abs_error:.2e} at step {q.first_violation_step}")
        print(f"         hint: {q.suggestion}")

Bug A (wrong actuator gain): passed = False
  CAUGHT actuator_torque: max_abs=5.00e-01 at step 0
         hint: Divergence is in an actuation quantity ('actuator_torque'); the actuator block in the candidate likely differs from the source (quantisation, friction model, or gain). Inspect the actuator template parameters.


In [5]:
# Bug B: an integration error — drift that grows over the trajectory (a bad
# integrator step on the position channel).
buggy2 = {k: v.copy() for k, v in reference.items()}
n = buggy2["joint_position"].shape[0]
buggy2["joint_position"] = buggy2["joint_position"] + np.linspace(0, 0.05, n)[:, None]
rep_b = compare(reference, buggy2, target_family="cortex-a76", dtype="float64")
print("Bug B (growing integration drift): passed =", rep_b.overall_passed)
for q in rep_b.per_quantity:
    if not q.passed:
        print(f"  CAUGHT {q.quantity}: max_abs={q.max_abs_error:.2e} at step {q.first_violation_step}")
        print(f"         hint: {q.suggestion}")

Bug B (growing integration drift): passed = False
  CAUGHT joint_position: max_abs=5.00e-02 at step 1
         hint: Early-step divergence with magnitude >> tolerance suggests an integration-error or numerical-instability issue; check the integrator settings and the dtype of the candidate run.


Both are caught, each with a hint pointed at the likely cause (an *actuation*
quantity → inspect the actuator template; an early-and-large divergence → an
integrator/dtype problem). This is the property that makes "it matches the
simulator" a claim you can *rely* on, not just assert.

## Step 5 — Cross-precision: why embedded bounds are looser

Embedded targets often run at `float32`. The reference is always computed at the
highest precision (`float64`), so a `float32` artifact is compared **cross-precision**
— its bound is `approximate` (engineering tolerance), not `ulp-bounded`, because
single-precision rounding accumulates over the horizon. Same target, different
precision, deliberately different contract:

In [6]:
for dt in ("float64", "float32"):
    try:
        t = lookup_tolerance("cortex-a76", dt, "joint_position")
        print(f"  cortex-a76 x {dt} x joint_position: abs={t.abs_tol:.0e} rel={t.rel_tol:.0e} [{t.grade}]")
    except KeyError:
        print(f"  cortex-a76 x {dt} x joint_position: (no row — would be a loud error)")

  cortex-a76 x float64 x joint_position: abs=1e-09 rel=1e-07 [ulp-bounded]
  cortex-a76 x float32 x joint_position: abs=5e-04 rel=2e-02 [approximate]


## Recap

HIL parity is the **trust model** under the whole stack:

- a host reference and the deployed artifact, run step-locked on identical inputs;
- compared against a **documented tolerance contract** (no magic numbers), with an
  intent grade per quantity;
- a robust **both-abs-and-rel** violation rule that avoids false alarms;
- and a check with **teeth** — it catches actuator bugs and integration drift, and
  points at the cause.

That's why, when tutorial #1 said *"the chip matches the simulator to ~4e-15,"* it
meant it. Next: **deploy a learned policy safely** — add an AI controller on top of
an MPC with a guaranteed on-target fallback, validated by exactly this harness.